# Splice site predictor 

Spliding enables the production of different isoforms from the same gene. Splice sites are conserved positions at the 5' and 3' end of introns. The 5' donor site conatins a GT dinucleotide, while the 3' acceptor site contains an AG dinucleotide. However, not all GT dinucleotides mark a donor site and not all AG dinucleotides mark an acceptor site. Here, I train a model to predict, which AG dinucleotides are donor sites an which are not.

## Download data

Download the latest fasta and gtf files for *Drosophila melanogaster*.

In [1]:
#!bash download_data.sh

SyntaxError: invalid syntax (230895574.py, line 5)

## Data processing

In [35]:
# get exon from gtf

import re

gtf_path="data/Drosophila_melanogaster.BDGP6.46.113.gtf"

exons: dict = {}

with open(gtf_path) as fh:
    for line in fh:
        if line.startswith('#'):
            continue
        parts = line.rstrip('\n').split('\t')    # get all fields from the gtf
        if len(parts) < 9 or parts[2] != 'exon':
            continue

        chrom  = parts[0]
        start  = int(parts[3]) - 1   # gtf is 1-based, make it 0-based
        end    = int(parts[4])        # gtf end is inclusive, make it exclusive
        strand = parts[6]

        m = re.search(r'transcript_id "([^"]+)"', parts[8])    #find transcript ID in attributes field and get it using regex
        if not m:    # if the transcript ID does not exist, don't use that row
            continue
        tid = m.group(1)

        key = (chrom, tid, strand)

        if key not in exons:    # set up key in dictionary
            exons[key] = []

        exons[key].append((start, end))    # add exon start and end position as value to dictionary
        # structure of exons, e.g. (('3R', 'FBtr0344474', '-'), [(17758708, 17758978), (17757023, 17757709)])



In [36]:
type(exons)

dict

In [ ]:
# get introns from exon positions
introns = []
for (chrom, tid, strand), exon_list in exons.items():
    exon_list.sort()    # sort by first element of tuple, i.e. exon start
    for i in range(len(exon_list) - 1):
        intron_start = exon_list[i][1]      # end of exon i
        intron_end   = exon_list[i + 1][0]  # start of exon i+1
        introns.append((chrom, intron_start, intron_end, strand))

In [9]:
for (chrom, tid, strand), exon_list in exons.items(): # check data
    exon_list.sort()
    print (chrom, tid, strand)

3R FBtr0344474 -
3R FBtr0083532 -
3R FBtr0346161 -
3R RR42085_transposable_element-RA -
3R FBtr0100373 -
3R FBtr0110781 +
3R FBtr0100842 +
3R FBtr0085326 +
3R FBtr0336764 +
3R FBtr0334905 -
3R FBtr0083540 +
3R FBtr0273265 +
3R FBtr0091615 +
3R FBti0063896-RA +
3R FBtr0085482 +
3R FBtr0081676 +
3R FBtr0083823 +
3R FBtr0344285 +
3R FBtr0085360 +
3R FBti0019428-RA -
3R FBti0063768-RA -
3R FBtr0085057 -
3R FBtr0113387 -
3R FBtr0084928 -
3R FBtr0084927 -
3R FBtr0113388 -
3R FBtr0113386 -
3R FBtr0113384 -
3R FBtr0336451 -
3R FBtr0113385 -
3R FBtr0336452 -
3R FBtr0084433 +
3R FBtr0332427 +
3R FBtr0301700 +
3R FBtr0300809 +
3R FBtr0300808 +
3R FBtr0472816 -
3R FBtr0113259 +
3R FBtr0084017 +
3R FBti0019461-RA +
3R FBtr0452128 +
3R FBtr0347496 +
3R FBtr0347308 +
3R FBtr0334915 -
3R FBti0059721-RA -
3R FBtr0082981 -
3R FBtr0334959 -
3R FBtr0333931 -
3R FBtr0084032 -
3R FBtr0330049 -
3R FBtr0083597 -
3R FBtr0083278 -
3R FBtr0344291 +
3R FBtr0290211 +
3R FBtr0083632 +
3R FBtr0321274 +
3R FBtr008568

In [6]:
print(introns[1:5]) # check data

[('3R', 17754245, 17754975, '-'), ('3R', 17755101, 17755181, '-'), ('3R', 17755284, 17757023, '-'), ('3R', 17757709, 17758708, '-')]


In [9]:
# install biopython if necessary
!pip install biopython

  Obtaining dependency information for biopython from https://files.pythonhosted.org/packages/f1/c7/06ae2e0672ef5eae35b5858355118cc265553d0ba8a23eaa1c056359683d/biopython-1.87-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/02/03/74fe2a4cb3817d94d86402f2506554130a2f01414e299b5a843e5a8a957f/numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.4 MB/s eta 0:00:000:00:0136m0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 10.9 MB/s eta 0:00:00m eta 0:00:010:01:01


In [25]:
# read fasta file and save as dict

fasta_path = "data/Drosophila_melanogaster.BDGP6.46.dna.toplevel.fa"

record_dict = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"))
#type(record_dict)
#print(record_dict.keys())    # chrom id
#print(record_dict.values())   # seq, chrom id, name, description, dbxrefs

In [42]:
# exploratory analysis of exon and intron lengths
import numpy as np

# exons
exon_lengths = []

for transcript, exon in exons.items():
    for start, end in exon:
        exon_lengths.append(end - start)

print(f"Number of exons: {len(exon_lengths)}")
print(f"Mean length:     {np.mean(exon_lengths):.1f}")
print(f"Median length:   {np.median(exon_lengths):.1f}")
print(f"Min length:      {min(exon_lengths)}")
print(f"Max length:      {max(exon_lengths)}")

#introns
intron_lengths = []

for chrom, start, end, strand in introns:
    intron_lengths.append(end - start)

print(f"Number of introns: {len(intron_lengths)}")
print(f"Mean length:     {np.mean(intron_lengths):.1f}")
print(f"Median length:   {np.median(intron_lengths):.1f}")
print(f"Min length:      {min(intron_lengths)}")
print(f"Max length:      {max(intron_lengths)}")

Number of exons: 196625
Mean length:     518.8
Median length:   253.0
Min length:      1
Max length:      66001
Number of introns: 155015
Mean length:     1625.8
Median length:   104.0
Min length:      2
Max length:      268107


In [45]:
# For testing, I first concentrate only on donor splice sites
# I will remove introns <10 bases

all_introns =        [end - start for _, start, end, strand in introns]
length_filtered =      [end - start for _, start, end, strand in introns if (end - start) >= 10]

print(f"All introns:            {len(all_introns)}")
print(f"After length filter:    {len(length_filtered)}")




All introns:            155015
After length filter:    155014


In [47]:
# extract sequences up and downstream of donor site (positives)

positives = []

for chrom, start, end, strand in introns:
    if (end - start) >= 10:
        chrom_seq = record_dict[chrom].seq
        
        if strand == "+":
            # intron start is the start coordinate
            seq = chrom_seq[max(0, start - 50): start + 50] # max in case the coordinate is <50
            positives.append(str(seq))
            
        elif strand == "-":
            # intron start is the end coordinate (gene is read right to left)
            # reverse complement to get the sequence in the correct orientation
            seq = chrom_seq[max(0, end - 50): end + 50].reverse_complement()
            positives.append(str(seq))

In [59]:
print(positives[:5])
print(len(positives))

['TCATTGCATCCAGGCACGACAATATGACCGACATAAGTGTTCACAACAAGGTAAGCTCCGTCGCAGCCATCCGCAACCATGTCCCAATGTCCGCATGTCC', 'ATCGTCTGGCTGCTCGCGTTGGCCATCACCTGTCCCCCCATGCTGGGATGGTAAGTACACTTGAAGAAATAATCAGTATGTAGGGTTATTTCCATAACGA', 'CTCCTCTGCACGGCATCCATTCTCAGCCTGTGCGCCATCAGTGTGGACAGGTGAGCATAGGCCTATCCAAAACATCCATAGTATTCCCCTGATATCGTAA', 'TCTCGTGGGCATCTTCGTCATGCCCCCCGCCGTCGCCGTCCATCTCATAGGTGAGTCTATGTGTGTATGAGTGTGAGTGTGTAAGTGTGTGTGTGAGTGT', 'CGTAGCAAGTGTTAAGTGTAAATAGTTCACAGATAAAAACACACATTGAGGTAAAAGCTATACACAAATTTTGAGATTCAAAATATATTAGGCACATTGA']
155014


In [61]:
# extract 100 bp sequences with GT dinucleotide in the middle which is not a donor site
import random

# extract intron start positions
intron_start_positions = set()
for chrom, start, end, strand in introns:
    if (end - start) >= 10:
        if strand == "+":
            intron_start_positions.add((chrom, start))
        elif strand == "-":
            intron_start_positions.add((chrom, end))  # biological start is at end coordinate


negatives = []
for chrom, record in record_dict.items():
    chrom_seq = str(record.seq)
    
    for i in range(len(chrom_seq) - 1):
        if chrom_seq[i:i+2] == "GT":
            if (chrom, i) not in intron_start_positions:
                neg_seq = chrom_seq[max(0, i - 50): i + 50]
                negatives.append(neg_seq)
print(len(negatives))

negatives_subsample = random.sample(negatives, len(positives))
print(len(negatives_subsample))
                

7469531
155014
